# Billboard Pop YouTube Dataset

Colab SSD에서 여러 곡을 병렬 다운로드·변환하고, 완성된 음원만 Google Drive에 저장합니다.

In [ ]:
%pip install -q -U "yt-dlp[default]"
!apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null
!curl -fsSL https://deno.land/install.sh | sh > /dev/null
import os
os.environ["PATH"] = "/root/.deno/bin:" + os.environ["PATH"]
print("setup complete")

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

SEED_PATH = Path("/content/drive/MyDrive/MOHIM/billboard_pop_genius_seed_recrawled.json")
if not SEED_PATH.is_file():
    raise FileNotFoundError(f"Google Drive seed JSON을 찾을 수 없습니다: {SEED_PATH}")

DATASET_DIR = Path("/content/drive/MyDrive/MOHIM/billboard_pop_dataset")
MAX_TRACKS = None    # 전체 seed 범위
RESET_FROM_INDEX = 1145  # 1145번 이후를 한 번 초기화하고 다시 다운로드
NUM_WORKERS = 4      # 빠른 병렬 처리; 403/429가 잦으면 2로 낮추기
SEARCH_RESULTS = 8
MAX_DURATION_SECONDS = 300.0
print(f"seed: {SEED_PATH}")
print(f"drive dataset: {DATASET_DIR}")

In [ ]:
from __future__ import annotations

import json
import re
import shutil
import tempfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any, Mapping

import yt_dlp

REJECTED_TERMS = ("cover", "instrumental", "karaoke", "live", "nightcore", "remix", "slowed", "sped up")
INSIGNIFICANT_WORDS = {"and", "audio", "feat", "featuring", "ft", "official", "the", "video"}
AUDIO_DIR = DATASET_DIR / "audio"
MANIFEST_PATH = DATASET_DIR / "tracks.json"
BACKUP_PATH = DATASET_DIR / "tracks.backup.json"
STAGING_PATH = DATASET_DIR / "tracks.next.json"
BACKUP_STAGING_PATH = DATASET_DIR / "tracks.backup.next.json"
WORK_DIR = Path("/content/mohim_billboard_work")
LOCAL_MANIFEST_PATH = WORK_DIR / "tracks.json"
RESET_COMPLETED_FROM_INDEX = None
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

def read_json(path: Path) -> dict[str, Any]:
    data = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(data, Mapping) or not isinstance(data.get("tracks"), list):
        raise ValueError(f"JSON must contain a tracks list: {path}")
    return dict(data)

def validated_manifest(path: Path) -> dict[str, Any]:
    if not path.is_file() or path.stat().st_size <= 0:
        raise ValueError(f"manifest is missing or empty: {path}")
    return read_json(path)

def replace_drive_file(staging: Path, destination: Path) -> None:
    try:
        staging.replace(destination)
    except OSError:
        shutil.copy2(staging, destination)
        staging.unlink(missing_ok=True)

def write_manifest(tracks: dict[str, dict[str, Any]], errors: list[dict[str, str]], skipped: list[dict[str, Any]]) -> None:
    payload = {
        "metadata": {
            "name": "mohim_billboard_pop_youtube_dataset",
            "updated_at": datetime.now(timezone.utc).isoformat(),
            "num_tracks": len(tracks),
            "num_errors": len(errors),
            "num_skipped": len(skipped),
            "max_duration_seconds": MAX_DURATION_SECONDS,
            "reset_completed_from_index": RESET_COMPLETED_FROM_INDEX,
        },
        "tracks": sorted(tracks.values(), key=lambda item: str(item.get("seed_id", ""))),
        "errors": errors,
        "skipped": skipped,
    }
    serialized = json.dumps(payload, ensure_ascii=False, indent=2) + "\n"
    LOCAL_MANIFEST_PATH.write_text(serialized, encoding="utf-8")
    validated_manifest(LOCAL_MANIFEST_PATH)
    shutil.copy2(LOCAL_MANIFEST_PATH, STAGING_PATH)
    validated_manifest(STAGING_PATH)
    try:
        validated_manifest(MANIFEST_PATH)
    except Exception:
        pass
    else:
        shutil.copy2(MANIFEST_PATH, BACKUP_STAGING_PATH)
        validated_manifest(BACKUP_STAGING_PATH)
        replace_drive_file(BACKUP_STAGING_PATH, BACKUP_PATH)
    replace_drive_file(STAGING_PATH, MANIFEST_PATH)
    validated_manifest(MANIFEST_PATH)

def load_previous_manifest() -> dict[str, Any]:
    for path in (MANIFEST_PATH, BACKUP_PATH):
        try:
            return validated_manifest(path)
        except Exception as exc:
            print(f"manifest unavailable: {path.name}: {type(exc).__name__}")
    return {"metadata": {}, "tracks": [], "errors": [], "skipped": []}

def clean_title(value: str) -> str:
    title = re.sub(r"\s*[\[(](official|lyrics?|audio|video|mv).*?[\])]\s*", " ", value, flags=re.I)
    return re.sub(r"\s+", " ", title).strip()

def words(value: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", clean_title(value).casefold())

def significant_words(value: str) -> set[str]:
    return {word for word in words(value) if len(word) >= 3 and word not in INSIGNIFICANT_WORDS}

def first_entries(info: Mapping[str, Any]) -> list[dict[str, Any]]:
    entries = info.get("entries")
    if entries is None:
        return [dict(info)]
    return [dict(entry) for entry in entries if isinstance(entry, Mapping)]

def search_youtube(query: str) -> list[dict[str, Any]]:
    options = {
        "extract_flat": True,
        "ignoreerrors": True,
        "noplaylist": True,
        "quiet": True,
        "noprogress": True,
        "no_warnings": True,
    }
    with yt_dlp.YoutubeDL(options) as downloader:
        info = downloader.extract_info(f"ytsearch{SEARCH_RESULTS}:{query}", download=False)
    return first_entries(info) if isinstance(info, Mapping) else []

def validate_candidate(artist: str, title: str, info: Mapping[str, Any]) -> tuple[bool, str]:
    youtube_title = str(info.get("title") or "").strip()
    normalized = " ".join(words(youtube_title))
    rejected = next((term for term in REJECTED_TERMS if re.search(rf"\b{re.escape(term)}\b", normalized)), None)
    if rejected:
        return False, f"excluded term: {rejected}"
    duration = info.get("duration")
    if duration is not None and not 90.0 <= float(duration) <= MAX_DURATION_SECONDS:
        return False, "duration outside allowed range"
    expected = significant_words(title) or set(words(title))
    candidate = significant_words(youtube_title) or set(words(youtube_title))
    overlap = len(expected & candidate) / max(1, len(expected))
    similarity = SequenceMatcher(None, " ".join(words(title)), " ".join(words(youtube_title))).ratio()
    artist_words = significant_words(artist)
    uploader_words = significant_words(str(info.get("uploader") or info.get("channel") or ""))
    if overlap < 0.6 and similarity < 0.5:
        return False, "title mismatch"
    if artist_words and not artist_words & (candidate | uploader_words):
        return False, "artist mismatch"
    return True, ""

def youtube_url(info: Mapping[str, Any]) -> str:
    url = str(info.get("webpage_url") or info.get("original_url") or info.get("url") or "").strip()
    if url.startswith("http"):
        return url
    video_id = str(info.get("id") or url).strip()
    return f"https://www.youtube.com/watch?v={video_id}" if video_id else ""

def download_audio(url: str) -> dict[str, Any]:
    with tempfile.TemporaryDirectory(prefix="mohim-youtube-", dir=WORK_DIR) as temporary:
        temp_dir = Path(temporary)
        options = {
            "format": "bestaudio/best",
            "extractor_args": {"youtube": {"player_client": ["web_embedded"]}},
            "outtmpl": str(temp_dir / "%(id)s.%(ext)s"),
            "noplaylist": True,
            "quiet": True,
            "noprogress": True,
            "no_warnings": True,
            "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "m4a", "preferredquality": "256"}],
        }
        with yt_dlp.YoutubeDL(options) as downloader:
            raw_info = downloader.extract_info(url, download=True)
        if not isinstance(raw_info, Mapping):
            raise RuntimeError("yt-dlp returned no metadata")
        entries = first_entries(raw_info)
        info = entries[0] if entries else dict(raw_info)
        video_id = str(info.get("id") or "").strip()
        candidates = sorted(temp_dir.glob(f"{video_id}.*"))
        if not video_id or not candidates:
            raise FileNotFoundError("downloaded audio was not produced")
        destination = AUDIO_DIR / f"{video_id}{candidates[0].suffix.lower()}"
        if destination.exists():
            candidates[0].unlink()
        else:
            shutil.move(str(candidates[0]), destination)
        info["drive_audio_path"] = str(destination)
        return info

def process_track(index: int, total: int, seed_track: Mapping[str, Any]) -> tuple[int, dict[str, Any] | None, dict[str, Any] | None, str]:
    seed_id = str(seed_track.get("id") or f"seed-{index}")
    artist = str(seed_track.get("artist") or "").strip()
    title = str(seed_track.get("title") or "").strip()
    lyrics = str(seed_track.get("lyrics") or "").strip()
    try:
        if not artist or not title or not lyrics:
            raise ValueError("seed track must contain artist, title, and lyrics")
        query = f"{artist} {title} official audio"
        failures = []
        downloaded = None
        selected_url = ""
        for candidate in search_youtube(query):
            accepted, reason = validate_candidate(artist, title, candidate)
            if not accepted:
                failures.append(reason)
                continue
            selected_url = youtube_url(candidate)
            if not selected_url:
                continue
            try:
                downloaded = download_audio(selected_url)
                break
            except Exception as exc:
                failures.append(f"{type(exc).__name__}: {exc}")
        if downloaded is None:
            raise RuntimeError("no downloadable matching result; " + " | ".join(failures[-5:]))
        duration = downloaded.get("duration")
        if duration is not None and float(duration) > MAX_DURATION_SECONDS:
            Path(downloaded["drive_audio_path"]).unlink(missing_ok=True)
            skipped = {"seed_id": seed_id, "artist": artist, "title": title, "reason": "duration exceeds limit"}
            return index, None, skipped, f"[{index}/{total}] {artist} - {title}: skipped"
        track = {
            "track_id": seed_id,
            "seed_id": seed_id,
            "youtube_id": str(downloaded.get("id") or ""),
            "source_url": selected_url,
            "audio_path": downloaded["drive_audio_path"],
            "artist": artist,
            "title": title,
            "genres": list(seed_track.get("genres") or ["Pop"]),
            "language": str(seed_track.get("language") or "English"),
            "lyrics": lyrics,
            "lyrics_source": seed_track.get("lyrics_source") or "genius",
            "lyrics_source_url": seed_track.get("lyrics_source_url"),
            "duration_seconds": duration,
            "youtube_title": downloaded.get("title"),
            "youtube_search_query": query,
        }
        return index, track, None, f"[{index}/{total}] {artist} - {title}: downloaded"
    except Exception as exc:
        error = {"seed_id": seed_id, "artist": artist, "title": title, "error": f"{type(exc).__name__}: {exc}"}
        return index, None, error, f"[{index}/{total}] {artist} - {title}: failed: {type(exc).__name__}: {exc}"

seed = read_json(SEED_PATH)
candidates = list(seed["tracks"])
if MAX_TRACKS is not None:
    candidates = candidates[:MAX_TRACKS]
previous = load_previous_manifest()
RESET_COMPLETED_FROM_INDEX = previous.get("metadata", {}).get("reset_completed_from_index")
existing = {str(item.get("seed_id") or item.get("track_id") or item.get("id")): dict(item) for item in previous.get("tracks", [])}
errors = list(previous.get("errors", []))
skipped = list(previous.get("skipped", []))
if RESET_FROM_INDEX is not None and RESET_COMPLETED_FROM_INDEX != RESET_FROM_INDEX:
    if not 1 <= RESET_FROM_INDEX <= len(candidates):
        raise ValueError(f"RESET_FROM_INDEX must be between 1 and {len(candidates)}")
    reset_ids = {str(track.get("id") or f"seed-{index}") for index, track in enumerate(candidates, 1) if index >= RESET_FROM_INDEX}
    removed = [existing.pop(seed_id) for seed_id in list(existing) if seed_id in reset_ids]
    removed_audio = 0
    for item in removed:
        youtube_id = str(item.get("youtube_id") or "").strip()
        if not youtube_id:
            continue
        for extension in (".m4a", ".mp3", ".webm", ".wav", ".flac", ".ogg"):
            audio_path = AUDIO_DIR / f"{youtube_id}{extension}"
            if audio_path.is_file():
                audio_path.unlink()
                removed_audio += 1
    errors = [item for item in errors if str(item.get("seed_id") or "") not in reset_ids]
    skipped = [item for item in skipped if str(item.get("seed_id") or "") not in reset_ids]
    RESET_COMPLETED_FROM_INDEX = RESET_FROM_INDEX
    write_manifest(existing, errors, skipped)
    print(f"reset complete from {RESET_FROM_INDEX}: {len(removed)} tracks / {removed_audio} audio files removed")
pending = []
total = len(candidates)
for index, seed_track in enumerate(candidates, 1):
    if RESET_FROM_INDEX is not None and index < RESET_FROM_INDEX:
        continue
    seed_id = str(seed_track.get("id") or f"seed-{index}")
    prior = existing.get(seed_id)
    prior_audio = AUDIO_DIR / f"{prior.get('youtube_id')}.m4a" if prior and prior.get("youtube_id") else None
    if prior and prior_audio and prior_audio.is_file():
        prior["audio_path"] = str(prior_audio)
        print(f"[{index}/{total}] {seed_track.get('artist')} - {seed_track.get('title')}: already downloaded")
    else:
        pending.append((index, seed_track))
write_manifest(existing, errors, skipped)

executor = ThreadPoolExecutor(max_workers=NUM_WORKERS)
futures = {executor.submit(process_track, index, total, track): index for index, track in pending}
try:
    for future in as_completed(futures):
        _, track, issue, message = future.result()
        if track is not None:
            existing[str(track["seed_id"])] = track
        elif issue is not None and "error" in issue:
            errors.append(issue)
        elif issue is not None:
            skipped.append(issue)
        write_manifest(existing, errors, skipped)
        print(message, flush=True)
except KeyboardInterrupt:
    for future in futures:
        future.cancel()
    print("interrupted; completed tracks are saved")
finally:
    executor.shutdown(wait=True, cancel_futures=True)
    write_manifest(existing, errors, skipped)

print(f"complete: {len(existing)} tracks / {len(errors)} errors / {len(skipped)} skipped")